# 예제 III-B · Feature-Based KD (수강생용)

같은 초기 가중치와 같은 데이터로 두 Student를 학습합니다. 하나는 정답만 사용하고, 다른 하나는 Teacher의 중간 Feature Map도 사용합니다. `FEATURE_WEIGHT`를 바꿔 차이를 살펴봅니다.

## 1. 실행 환경과 CIFAR-10

처음부터 순서대로 실행합니다. CUDA가 있으면 GPU를 사용하며, CIFAR-10은 캐시가 없는 런타임에서 내려받습니다.

기본 실습은 **학습 16배치, 평가 512장**입니다. 난수 시드를 고정해 테스트 샘플을 선택하고, 학습 배치와 평가 데이터를 분리합니다.

**코드 셀 1 · 실행 환경과 학습 설정**

실행 장치와 데이터 경로를 준비하고 Student별 학습 16배치와 테스트 512장을 설정합니다.

In [ ]:
# 실습 설정을 한곳에서 관리한다. 데이터·가중치 캐시를 재사용하고 CUDA가 있으면 GPU로 실행한다.
import copy
import os
import time
from itertools import islice
from pathlib import Path
from statistics import mean

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# 동일한 시드와 배치 수를 사용해 반복 실행 조건을 맞춘다.
SEED = 50
BATCH_SIZE = 64
TRAIN_STEPS = 16
TEST_SAMPLES = 512
DATA_ROOT = Path(os.environ.get("AX_IDEC_DATA_ROOT", "./data"))
ASSET_ROOT = Path(os.environ.get("AX_IDEC_CACHE", "./idec_student_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("실행 장치:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

**코드 셀 2 · Teacher 기준으로 CIFAR-10 준비**

공개 Teacher에 맞춘 RGB 정규화를 적용하고 두 Student에 공통으로 사용할 학습과 테스트 데이터를 준비합니다.

이미 내려받은 CIFAR-10은 런타임의 캐시에서 읽습니다. 첫 실행에서는 공유 Google Drive에서 받고, 실패하면 CIFAR-10의 공식 주소를 사용합니다.

In [ ]:
# CIFAR-10 학습·테스트 분할에 같은 RGB 정규화를 적용한다. 테스트는 학습에 쓰이지 않는다.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
from http.client import HTTPException
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from urllib.request import urlopen

from torchvision.datasets.utils import check_integrity


class DriveCIFAR10(datasets.CIFAR10):
    # 압축 해제된 데이터나 검증된 압축 파일을 먼저 재사용한다.
    def download(self):
        if self._check_integrity():
            return

        root = Path(self.root)
        archive = root / self.filename
        if not check_integrity(archive, self.tgz_md5):
            drive_url = (
                "https://drive.usercontent.google.com/download"
                "?id=1Sx382Bt_XcAUj9_igoCgWoNQ6c5ooSa6&export=download&confirm=t"
            )
            try:
                root.mkdir(parents=True, exist_ok=True)
                with TemporaryDirectory(dir=root) as temp_dir:
                    staged = Path(temp_dir) / self.filename
                    print("Google Drive에서 CIFAR-10을 내려받습니다.")
                    with urlopen(drive_url, timeout=30) as response:
                        if response.headers.get_content_type() == "text/html":
                            raise ValueError("Drive가 파일 대신 안내 페이지를 반환했습니다.")
                        with staged.open("wb") as output:
                            copyfileobj(response, output, length=1024 * 1024)
                    if not check_integrity(staged, self.tgz_md5):
                        raise ValueError("CIFAR-10 원본 체크섬과 일치하지 않습니다.")
                    staged.replace(archive)
            except (OSError, HTTPException, ValueError):
                print("Drive에서 CIFAR-10을 받지 못해 공식 다운로드 주소로 다시 시도합니다.")

        # torchvision이 원본 체크섬을 확인하고 필요한 경우 압축을 해제한다.
        super().download()


train_data = DriveCIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_data = DriveCIFAR10(DATA_ROOT, train=False, download=True, transform=transform)

# 시드를 고정해 일부 샘플만 선택하고, 같은 데이터에서 반복 비교할 Loader를 만든다.
def make_loader(dataset, samples, seed):
    indices = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(seed))
    subset = Subset(dataset, indices[:samples].tolist())
    return DataLoader(subset, batch_size=BATCH_SIZE, num_workers=0,
                      pin_memory=(DEVICE.type == "cuda"))

train_loader = make_loader(train_data, (TRAIN_STEPS + 1) * BATCH_SIZE, SEED)
test_loader = make_loader(test_data, TEST_SAMPLES, SEED + 1)
print(f"학습 {TRAIN_STEPS} batches / 평가 {len(test_loader.dataset)} images")

## 2. 공통 Student 모델

경량 ResNet-18은 BasicBlock의 공간 연산을 Depthwise 3×3과 Pointwise 1×1로 나눕니다. 입력은 32×32 RGB 영상이고, 분류기는 CIFAR-10 클래스별 Logit 10개를 출력합니다.

학습 방법에 따라 달라지는 것은 **Student가 정답 외에 함께 받는 학습 신호**입니다.

**코드 셀 3 · Student 모델 구조 정의**

Feature-Based KD 실습에 사용할 Depthwise Separable ResNet-18을 정의합니다.

In [ ]:
# 기존 BasicBlock의 3×3 연산을 Depthwise와 Pointwise로 나누어 경량 모델을 만든다.
def build_light_resnet18():
    __all__ = ['ResNet', 'resnet18']

    def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
        """3x3 convolution with padding"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                         padding=dilation, groups=in_planes, bias=False, dilation=dilation)

    def conv1x1(in_planes, out_planes, stride=1):
        """1x1 convolution"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

    # 주 경로에서 특징을 계산하고 Residual 입력을 더한다. 크기가 바뀌는 블록에서는 Residual도 변환한다.
    class BasicBlock(nn.Module):
        expansion = 1

        def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                     base_width=64, dilation=1, norm_layer=None):
            super(BasicBlock, self).__init__()

            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            if groups != 1 or base_width != 64:
                raise ValueError(
                    'BasicBlock only supports groups=1 and base_width=64')
            if dilation > 1:
                raise NotImplementedError(
                    "Dilation > 1 not supported in BasicBlock")
            # Both self.conv1 and self.downsample layers downsample the input when stride != 1
            # 3×3 Depthwise에서 채널별 공간 특징을 계산하고, 1×1 Pointwise에서 채널을 조합한다.
            self.dwconv33_1 = conv3x3(inplanes, planes, stride)
            self.conv11_1 = conv1x1(planes, planes)
            self.bn1 = norm_layer(planes)
            self.relu = nn.ReLU(inplace=True)

            self.dwconv33_2 = conv3x3(planes, planes)
            self.conv11_2 = conv1x1(planes, planes)
            self.bn2 = norm_layer(planes)
            self.downsample = downsample
            self.stride = stride

        def forward(self, x):
            identity = x

            out = self.dwconv33_1(x)
            out = self.conv11_1(out)
            out = self.bn1(out)
            out = self.relu(out)

            out = self.dwconv33_2(out)
            out = self.conv11_2(out)
            out = self.bn2(out)

            if self.downsample is not None:
                identity = self.downsample(x)

            out += identity
            out = self.relu(out)

            return out

    # 입력 stem과 Residual stage를 연결하고, Global Average Pooling 뒤에서 클래스별
    # Logit을 계산한다.
    class ResNet(nn.Module):

        def __init__(self, block, layers, num_classes=1000, zero_init_residual=False,
                     groups=1, width_per_group=64, replace_stride_with_dilation=None,
                     norm_layer=None):
            super(ResNet, self).__init__()
            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            self._norm_layer = norm_layer

            self.inplanes = 64
            self.dilation = 1
            if replace_stride_with_dilation is None:
                # each element in the tuple indicates if we should replace
                # the 2x2 stride with a dilated convolution instead
                replace_stride_with_dilation = [False, False, False]
            if len(replace_stride_with_dilation) != 3:
                raise ValueError("replace_stride_with_dilation should be None "
                                 "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
            self.groups = groups
            self.base_width = width_per_group
            self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                                   bias=False)

            self.bn1 = norm_layer(self.inplanes)
            self.relu = nn.ReLU(inplace=True)
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            # 네 stage는 64·128·256·512채널을 사용한다. 첫 블록의 stride로 해상도를 조정한다.
            self.layer1 = self._make_layer(block, 64, layers[0])
            self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                           dilate=replace_stride_with_dilation[0])
            self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                           dilate=replace_stride_with_dilation[1])
            self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                           dilate=replace_stride_with_dilation[2])
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
            self.fc = nn.Linear(512 * block.expansion, num_classes)

            # Convolution과 정규화 계층의 학습 초기값을 설정한다.
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)

            # Zero-initialize the last BN in each residual branch,
            # so that the residual branch starts with zeros, and each residual block behaves like an identity.
            # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
            if zero_init_residual:
                for m in self.modules():
                    if isinstance(m, Bottleneck):
                        nn.init.constant_(m.bn3.weight, 0)
                    elif isinstance(m, BasicBlock):
                        nn.init.constant_(m.bn2.weight, 0)

        # 첫 블록에서 필요한 Residual 변환을 적용하고, 뒤의 블록은 같은 출력 크기로 이어 붙인다.
        def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
            norm_layer = self._norm_layer
            downsample = None
            previous_dilation = self.dilation
            if dilate:
                self.dilation *= stride
                stride = 1
            if stride != 1 or self.inplanes != planes * block.expansion:
                downsample = nn.Sequential(
                    conv1x1(self.inplanes, planes * block.expansion, stride),
                    norm_layer(planes * block.expansion),
                )

            layers = []
            layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                                self.base_width, previous_dilation, norm_layer))
            self.inplanes = planes * block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, planes, groups=self.groups,
                                    base_width=self.base_width, dilation=self.dilation,
                                    norm_layer=norm_layer))

            return nn.Sequential(*layers)

        # 입력에서 stem, 네 stage, Pooling, 최종 Linear Layer로 이어지는 순전파다.
        def _forward_impl(self, x):
            # See note [TorchScript super()]
            x = self.conv1(x)
            x = self.bn1(x)
            x = self.relu(x)
            x = self.maxpool(x)

            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)

            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.fc(x)

            return x

        def forward(self, x):
            return self._forward_impl(x)

    # CIFAR-10에 맞게 첫 Conv2d를 3×3, stride 1로 설정하고 출력 클래스 수를 10으로 둔다.
    def resnet_18_cifar_lightweight():
        r"""
            Modify the resnet 18 network in order to run on cifar-10 dataset

            To enhance the accuracy, the (kernel_size, stride, padding) of conv1 is modified to (3, 1, 1)
            referenced by <https://github.com/akamaster/pytorch_resnet_cifar10>
        """
        model = ResNet(block=BasicBlock, layers=[2, 2, 2, 2], num_classes=10)

        model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1,
                                bias=False)

        return model

    return resnet_18_cifar_lightweight()

**코드 셀 4 · 정확도 평가 함수 정의**

모델의 예측 클래스를 정답과 비교해 테스트 정확도를 계산하는 함수를 정의합니다.

In [ ]:
# 평가 모드에서 각 영상의 최댓값 클래스를 정답과 비교해 전체 정확도를 집계한다.
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += labels.numel()
    return 100 * correct / total

## 3. 고정 Teacher 준비

[CIFAR-10 ResNet-18 공개 체크포인트](https://huggingface.co/edadaltocg/resnet18_cifar10)를 다운로드하거나 기존 캐시에서 불러옵니다. Teacher를 평가 모드에 두고 파라미터의 gradient를 끕니다. Student를 학습하는 동안 Teacher의 출력은 고정된 학습 목표로 사용됩니다.

표시되는 **Teacher 정확도**는 체크포인트와 입력 전처리를 확인하는 기준입니다.

**코드 셀 5 · 사전 학습 Teacher 고정**

CIFAR-10 사전 학습 가중치를 Teacher에 불러 평가 모드로 고정하고 테스트 정확도를 출력합니다.

In [ ]:
from torchvision.models import resnet18

# CIFAR-10으로 사전 학습한 공개 Teacher의 체크포인트를 불러온다.
TEACHER_URL = ("https://huggingface.co/edadaltocg/resnet18_cifar10/resolve/"
               "ccacbae365101b9bfa45e1f17f4acac30c2601b2/pytorch_model.bin")
teacher = resnet18(weights=None, num_classes=10)
teacher.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
teacher.maxpool = nn.Identity()
state_dict = torch.hub.load_state_dict_from_url(
    TEACHER_URL, model_dir=str(ASSET_ROOT), file_name="teacher.bin",
    map_location="cpu", weights_only=True
)
teacher.load_state_dict(state_dict, strict=True)
# Offline KD에서는 Teacher를 평가 모드에 두고 가중치를 고정한다.
teacher = teacher.to(DEVICE).eval().requires_grad_(False)
print(f"Teacher 정확도: {evaluate(teacher, test_loader):.2f}%")

## 4. Feature Map 추출과 Feature KD Loss

셀 6–7에서 Teacher와 Student의 중간 Feature Map을 추출하고, 공간 크기를 맞춘 뒤 MSE를 계산하는 함수를 준비합니다.

**코드 셀 6 · layer2의 Feature Map 추출**

입력 이미지에서 layer2의 Feature Map을 추출하고 이를 이어서 분류에 사용하는 함수를 정의합니다.

In [ ]:
# 입력에서 layer2까지의 중간 특징을 꺼낸다. 같은 Feature Map을 분류 계산에도 재사용할 수 있다.
def features_at_layer2(model, images):
    x = model.conv1(images)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)
    x = model.layer1(x)
    return model.layer2(x)

# 중간 Feature Map을 후속 stage와 최종 분류기에 전달한다.
def predict_from_layer2(model, feature):
    x = model.layer3(feature)
    x = model.layer4(x)
    x = torch.flatten(model.avgpool(x), 1)
    return model.fc(x)

**코드 셀 7 · Feature KD Loss 계산**

Teacher의 Feature Map을 Student의 공간 크기에 맞추고 두 Feature Map의 평균 제곱 오차를 계산합니다.

In [ ]:
# Teacher의 공간 크기만 Student에 맞춘 뒤, 같은 위치의 활성값 차이를 MSE로 계산한다.
def feature_kd(student_feature, teacher_feature):
    # Teacher의 공간 해상도를 Student의 Feature Map 크기에 맞춥니다.
    target = F.adaptive_avg_pool2d(teacher_feature.detach(), student_feature.shape[-2:])
    return F.mse_loss(student_feature, target)

### TODO · Feature loss 가중치 바꾸기

- **수정할 값** · 바로 아래 **코드 셀 8**에서 `FEATURE_WEIGHT`를 **0**으로 두고 셀 8–10을 실행한 뒤, **0.2**로 바꾸고 셀 8–10을 다시 실행합니다. 셀 8에서 두 Student를 다시 초기화합니다.
- **실행 후 확인** · 셀 9에서 두 Student의 정확도를, 셀 10에서 같은 이미지의 Feature Map과 Teacher 대비 MSE를 비교합니다.

**코드 셀 8 · Feature KD 비중과 초기 조건 설정**

FEATURE_WEIGHT를 설정하고 두 Student를 같은 가중치로 초기화한 뒤 Teacher와 Student의 Feature Map 크기를 확인합니다.

In [ ]:
# @title Feature KD 비중 선택
FEATURE_WEIGHT = 0.2 # @param {type:"slider", min:0, max:1, step:0.1}

# 매 실행에서 두 Student를 같은 가중치로 초기화한다.
torch.manual_seed(SEED)
student_ce = build_light_resnet18().to(DEVICE)
student_feature_kd = copy.deepcopy(student_ce)
sample = next(iter(test_loader))[0][:1].to(DEVICE)
student_ce.eval()
with torch.inference_mode():
    teacher_feature = features_at_layer2(teacher, sample)
    student_feature = features_at_layer2(student_ce, sample)
    aligned_teacher = F.adaptive_avg_pool2d(teacher_feature, student_feature.shape[-2:])
print("Teacher:", tuple(teacher_feature.shape))
print("Student:", tuple(student_feature.shape))
print("정렬 후 Teacher:", tuple(aligned_teacher.shape))

## 5. 같은 조건에서 두 Student 비교

셀 9는 두 모델을 같은 16배치로 학습해 테스트 정확도를 출력합니다. 셀 10에서는 같은 이미지·채널·색 범위의 Feature Map을 나란히 표시하고, 그 이미지에서 Teacher와의 평균 제곱 오차도 확인합니다.

**코드 셀 9 · 두 Student 학습과 정확도 비교**

같은 16배치로 두 Student를 학습해 정답만 사용한 경우와 Feature KD를 함께 적용한 경우의 테스트 정확도를 비교합니다.

In [ ]:
# Teacher는 고정하고 두 Student에게 같은 배치를 순서대로 전달한다.
student_ce.train()
student_feature_kd.train()
optim_ce = torch.optim.SGD(student_ce.parameters(), lr=0.03, momentum=0.9)
optim_feature = torch.optim.SGD(student_feature_kd.parameters(), lr=0.03, momentum=0.9)

for images, labels in islice(train_loader, TRAIN_STEPS):
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    with torch.no_grad():
        target = features_at_layer2(teacher, images)

    optim_ce.zero_grad(set_to_none=True)
    label_loss = F.cross_entropy(student_ce(images), labels)
    label_loss.backward()
    optim_ce.step()

    optim_feature.zero_grad(set_to_none=True)
    student_map = features_at_layer2(student_feature_kd, images)
    logits = predict_from_layer2(student_feature_kd, student_map)
    ce = F.cross_entropy(logits, labels)
    kd = feature_kd(student_map, target)
    (ce + FEATURE_WEIGHT * kd).backward()
    optim_feature.step()

acc_ce = evaluate(student_ce, test_loader)
acc_feature = evaluate(student_feature_kd, test_loader)
print(f"학습 {TRAIN_STEPS}배치 | 평가 {len(test_loader.dataset)}장")
print(f"정답만 학습: {acc_ce:.2f}%")
print(f"정답 + Feature KD (비중 {FEATURE_WEIGHT:g}): {acc_feature:.2f}%")
print(f"정확도 차이: {acc_feature - acc_ce:+.2f}%p")

**코드 셀 10 · Feature Map과 MSE 비교**

같은 이미지와 채널에서 Teacher와 두 Student의 Feature Map을 같은 색 범위로 표시하고 Teacher 대비 MSE를 출력합니다.

In [ ]:
import matplotlib.pyplot as plt

# 세 화면에 같은 이미지, 채널 번호, 색 범위를 적용한다.
channel = 0
student_ce.eval()
student_feature_kd.eval()
with torch.inference_mode():
    ce_map = features_at_layer2(student_ce, sample)
    kd_map = features_at_layer2(student_feature_kd, sample)
    ce_error = F.mse_loss(ce_map, aligned_teacher).item()
    kd_error = F.mse_loss(kd_map, aligned_teacher).item()
    maps = [feature[0, channel].cpu() for feature in (aligned_teacher, ce_map, kd_map)]

limit = max(float(m.abs().max()) for m in maps) or 1.0
titles = ["Teacher (aligned)", "Student: CE only", f"Student: CE + Feature KD ({FEATURE_WEIGHT:g})"]
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), constrained_layout=True)
for axis, feature, title in zip(axes, maps, titles):
    heatmap = axis.imshow(feature, vmin=-limit, vmax=limit, cmap="coolwarm")
    axis.set_title(title)
    axis.axis("off")
fig.colorbar(heatmap, ax=axes, label=f"Activation (channel {channel})")
plt.show()
print(f"같은 입력 1장 · Teacher와의 Feature MSE | 정답만: {ce_error:.4f} | 정답 + Feature KD: {kd_error:.4f}")

## 추가로 바꿔볼 설정 · 선택 실습

- `FEATURE_WEIGHT`를 0, 0.2, 0.5로 바꿔 셀 8–10을 실행합니다. 같은 이미지에서 Teacher와의 Feature MSE와 테스트 정확도를 각각 비교합니다.
- 학습 조건을 고정한 채 표시할 `channel`만 다른 번호로 바꿔 셀 10을 실행합니다. 화면에서 강조되는 영역이 어떻게 달라지는지 확인합니다.